# Tata Motors Stock Market Trend Prediction
## HMM vs KNN vs AdaBoost | Google Colab Mini Project

**Dataset:** `TATAMOTORS.NS.csv` supplied for this project (8,427 historical rows).

### Project objective
Use historical Tata Motors stock data to classify the **next trading day's direction**:
- `1 = UP`: next day's adjusted closing price is greater than or equal to today's.
- `0 = DOWN`: next day's adjusted closing price is lower than today's.

### Algorithms
1. **Gaussian Hidden Markov Model (HMM):** discovers hidden market states from return and volatility sequences.
2. **K-Nearest Neighbors (KNN):** classifies a day based on similar historical feature patterns.
3. **AdaBoost:** combines shallow decision trees to classify the next-day direction.

> Educational project only. A classification result is not a guaranteed stock forecast or investment recommendation.


## 1. Install required libraries
Run this cell first in Google Colab. It installs HMM support; the other libraries are usually preinstalled in Colab.

In [ ]:
!pip -q install hmmlearn yfinance

## 2. Upload your Tata Motors CSV file
When the upload window appears, select `TATAMOTORS.NS.csv`. If you already placed the CSV in the Colab session, the notebook will also look for it by filename.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from google.colab import files

CSV_FILE = "TATAMOTORS.NS.csv"
if not os.path.exists(CSV_FILE):
    uploaded = files.upload()
    if CSV_FILE not in uploaded and len(uploaded) > 0:
        CSV_FILE = next(iter(uploaded.keys()))

raw = pd.read_csv(CSV_FILE)
print("Loaded file:", CSV_FILE)
print("Dataset shape:", raw.shape)
display(raw.head())


## 3. Inspect and clean the dataset
The uploaded file uses `Unnamed: 0` as its date column and lowercase OHLCV names. The code standardizes the column names automatically.

In [ ]:
# Standardize column names
raw.columns = [str(c).strip().lower().replace(" ", "") for c in raw.columns]

# Identify date column
date_candidates = [c for c in raw.columns if c in ["date", "datetime", "unnamed:0"]]
if not date_candidates:
    raise ValueError("Could not identify the date column. Expected Date or Unnamed: 0.")
date_col = date_candidates[0]

required = ["open", "high", "low", "close"]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required price columns: {missing}")

raw = raw.rename(columns={date_col: "Date"})
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
for col in ["open", "high", "low", "close", "adjclose", "volume"]:
    if col in raw.columns:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

if "adjclose" not in raw.columns:
    raw["adjclose"] = raw["close"]
if "volume" not in raw.columns:
    raw["volume"] = 0

raw = (raw.dropna(subset=["Date", "open", "high", "low", "close", "adjclose"])
          .sort_values("Date")
          .drop_duplicates(subset=["Date"], keep="last")
          .reset_index(drop=True))

print("Cleaned shape:", raw.shape)
print("Date range:", raw["Date"].min().date(), "to", raw["Date"].max().date())
print("Missing values:")
display(raw.isna().sum().to_frame("Missing values"))
display(raw.head())


## 4. Exploratory Data Analysis (EDA)
We plot adjusted closing price and trading volume to understand the historical data.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(raw["Date"], raw["adjclose"], linewidth=1)
axes[0].set_title("Tata Motors Adjusted Closing Price")
axes[0].set_ylabel("Adjusted price")
axes[0].grid(alpha=0.25)

axes[1].plot(raw["Date"], raw["volume"], linewidth=0.7)
axes[1].set_title("Tata Motors Trading Volume")
axes[1].set_ylabel("Volume")
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Feature engineering and target creation

Features are created using the current day's and previous days' information:
- Daily adjusted-price return
- Previous one-day and two-day returns
- 5-day / 10-day moving-average ratio
- 5-day rolling volatility
- Intraday high-low range
- Open-to-close return
- Trading-volume percentage change

The target is the next trading day's direction. Rows with unavailable rolling/lag features are removed.


In [ ]:
data = raw.copy()
price = data["adjclose"].where(data["adjclose"] > 0, data["close"])

data["Return"] = price.pct_change()
data["Return_Lag1"] = data["Return"].shift(1)
data["Return_Lag2"] = data["Return"].shift(2)
data["MA_5"] = price.rolling(5).mean()
data["MA_10"] = price.rolling(10).mean()
data["MA_Ratio"] = data["MA_5"] / data["MA_10"] - 1
data["Volatility"] = data["Return"].rolling(5).std()
data["Intraday_Range"] = (data["high"] - data["low"]) / price
data["Open_Close_Return"] = (data["close"] - data["open"]) / data["open"].replace(0, np.nan)

# Volume may be zero in some old records. Log-volume change is more stable.
data["Log_Volume"] = np.log1p(data["volume"].clip(lower=0))
data["Volume_Change"] = data["Log_Volume"].diff()

# 1 = next adjusted close UP or unchanged; 0 = next adjusted close DOWN
next_price = price.shift(-1)\ndata["Target"] = np.where(next_price.notna(), (next_price >= price).astype(float), np.nan)

FEATURES = [
    "Return", "Return_Lag1", "Return_Lag2", "MA_Ratio",
    "Volatility", "Intraday_Range", "Open_Close_Return", "Volume_Change"
]
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ["Target"]).reset_index(drop=True)

print("Rows after feature engineering:", len(data))
print("Features:", FEATURES)
print("Target balance (0=DOWN, 1=UP):")
display(data["Target"].value_counts().rename(index={0:"DOWN", 1:"UP"}).to_frame("Count"))
display(data[["Date"] + FEATURES + ["Target"]].head())


## 6. Chronological train-test split and scaling
The first 80% of the observations are used for training and the final 20% for testing. This preserves time order. The scaler is fitted only on training data.

In [ ]:
from sklearn.preprocessing import StandardScaler

split_index = int(len(data) * 0.80)
train = data.iloc[:split_index].copy()
test = data.iloc[split_index:].copy()

X_train = train[FEATURES]
X_test = test[FEATURES]
y_train = train["Target"].astype(int)
y_test = test["Target"].astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training records:", len(train))
print("Testing records:", len(test))
print("Training period:", train["Date"].min().date(), "to", train["Date"].max().date())
print("Testing period:", test["Date"].min().date(), "to", test["Date"].max().date())


## 7. Model 1 — K-Nearest Neighbors (KNN)
KNN compares each test observation with the seven nearest training observations. Distance weighting gives more influence to closer neighbors.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

knn = KNeighborsClassifier(n_neighbors=7, weights="distance")
knn.fit(X_train_scaled, y_train)
pred_knn = knn.predict(X_test_scaled)

print("KNN classification report")
print(classification_report(y_test, pred_knn, target_names=["DOWN", "UP"], zero_division=0))


## 8. Model 2 — AdaBoost
AdaBoost trains a sequence of shallow decision trees (decision stumps). It places more emphasis on observations that previous learners classified incorrectly.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

weak_learner = DecisionTreeClassifier(max_depth=1, random_state=42)
try:
    adaboost = AdaBoostClassifier(
        estimator=weak_learner, n_estimators=100,
        learning_rate=0.5, random_state=42
    )
except TypeError:  # Compatibility with older scikit-learn versions
    adaboost = AdaBoostClassifier(
        base_estimator=weak_learner, n_estimators=100,
        learning_rate=0.5, random_state=42
    )

adaboost.fit(X_train_scaled, y_train)
pred_ada = adaboost.predict(X_test_scaled)

print("AdaBoost classification report")
print(classification_report(y_test, pred_ada, target_names=["DOWN", "UP"], zero_division=0))


## 9. Model 3 — Hidden Markov Model (HMM)

A Gaussian HMM is trained on the training-period sequence of daily returns and volatility. It discovers three hidden states. State labels are arbitrary, so the notebook maps each state to the most frequent UP/DOWN target in the training data only.

For test decoding, the fitted HMM decodes the concatenated train and test feature sequence, then the test portion is selected. The HMM parameters are still fitted on training observations only.


In [ ]:
from hmmlearn.hmm import GaussianHMM

HMM_FEATURES = ["Return", "Volatility"]
hmm_scaler = StandardScaler()
hmm_train = hmm_scaler.fit_transform(train[HMM_FEATURES])
hmm_test = hmm_scaler.transform(test[HMM_FEATURES])

hmm = GaussianHMM(
    n_components=3,
    covariance_type="diag",
    n_iter=300,
    random_state=42
)
hmm.fit(hmm_train)

# Decode a continuous sequence so test state decoding follows the end of training sequence.
all_hmm_observations = np.vstack([hmm_train, hmm_test])
all_states = hmm.predict(all_hmm_observations)
states_train = all_states[:len(train)]
states_test = all_states[len(train):]

overall_majority = int(y_train.mode().iloc[0])
state_to_direction = {}
for state in range(hmm.n_components):
    state_targets = y_train.to_numpy()[states_train == state]
    if len(state_targets):
        state_to_direction[state] = int(pd.Series(state_targets).mode().iloc[0])
    else:
        state_to_direction[state] = overall_majority

pred_hmm = np.array([state_to_direction[s] for s in states_test])

print("Training-only hidden-state to direction mapping:", state_to_direction)
print("HMM classification report")
print(classification_report(y_test, pred_hmm, target_names=["DOWN", "UP"], zero_division=0))


## 10. Compare all three algorithms
All metrics are calculated on the same held-out chronological test set.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

predictions = {
    "HMM": pred_hmm,
    "KNN": pred_knn,
    "AdaBoost": pred_ada
}

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }

metrics = pd.DataFrame({
    model_name: get_metrics(y_test, pred)
    for model_name, pred in predictions.items()
}).T

display(metrics.style.format("{:.4f}"))
best_model = metrics["F1-score"].idxmax()
print("Best model by test F1-score:", best_model)


In [ ]:
ax = metrics.plot(kind="bar", figsize=(11, 5), ylim=(0, 1), rot=0)
ax.set_title("Tata Motors: HMM vs KNN vs AdaBoost")
ax.set_ylabel("Score")
ax.set_xlabel("Algorithm")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 11. Confusion matrices

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (model_name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["DOWN", "UP"]).plot(
        ax=ax, colorbar=False, values_format="d"
    )
    ax.set_title(model_name)
plt.tight_layout()
plt.show()


## 12. Actual vs predicted trend

In [ ]:
results = pd.DataFrame({
    "Date": test["Date"].to_numpy(),
    "Actual": y_test.to_numpy(),
    "HMM_Predicted": pred_hmm,
    "KNN_Predicted": pred_knn,
    "AdaBoost_Predicted": pred_ada
})
display(results.head(20).replace({0:"DOWN", 1:"UP"}))

n_plot = min(150, len(results))
plt.figure(figsize=(14, 5))
plt.step(results["Date"].iloc[:n_plot], results["Actual"].iloc[:n_plot],
         where="mid", label="Actual", linewidth=2)
plt.step(results["Date"].iloc[:n_plot], results["HMM_Predicted"].iloc[:n_plot],
         where="mid", label="HMM", alpha=0.8)
plt.step(results["Date"].iloc[:n_plot], results["KNN_Predicted"].iloc[:n_plot],
         where="mid", label="KNN", alpha=0.8)
plt.step(results["Date"].iloc[:n_plot], results["AdaBoost_Predicted"].iloc[:n_plot],
         where="mid", label="AdaBoost", alpha=0.8)
plt.title("Actual vs Predicted Tata Motors Trend (test sample)")
plt.xlabel("Date")
plt.ylabel("Direction (0=DOWN, 1=UP)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 13. Save and download project results
The notebook exports metrics and test predictions as CSV files and downloads them to your computer.

In [ ]:
import os
os.makedirs("outputs", exist_ok=True)

metrics.to_csv("outputs/tatamotors_model_comparison.csv")
results.to_csv("outputs/tatamotors_test_predictions.csv", index=False)

print("Saved files:")
print("- outputs/tatamotors_model_comparison.csv")
print("- outputs/tatamotors_test_predictions.csv")

from google.colab import files
files.download("outputs/tatamotors_model_comparison.csv")
files.download("outputs/tatamotors_test_predictions.csv")


## 14. Conclusion

This project compares three different machine-learning approaches on the supplied Tata Motors historical dataset:
- HMM models latent market states from a sequence of returns and volatility.
- KNN classifies a new observation using similar historical feature vectors.
- AdaBoost combines weak decision-tree learners into an ensemble classifier.

Use the metric table and confusion matrices generated above to discuss which algorithm performed best on this particular test period. Do not claim that the best test score guarantees future profitability.

### Limitations and future scope
- Historical price patterns do not guarantee future price movements.
- Market events, corporate actions, costs, slippage and liquidity are not modeled.
- HMM hidden states are statistical states, not direct labels of market sentiment.
- Future work can use walk-forward validation, multiple stocks, additional indicators and realistic backtesting.
